# DiscoCH: near-real-time vs. all-data season
Reruns a finished season with every Sentinel-2 observation and compares it with the near-real-time series from `DiscoCH.ipynb`:

- **Near real time (5-day lag):** each date is interpolated from observations up to 5 days after it (`DiscoCH.ipynb`).
- **All data:** each date is interpolated from all observations within the RBF kernel, before and after it.

Both are normalized with the same stepwise min/max: the near-real-time run's per-date snapshots (`minmax_by_date/`), so nothing after a date enters its min/max and only the interpolation differs.

*Run `DiscoCH.ipynb` up to and including Masking first (with `export_minmax=True`), at the end of the season.*

In [ ]:
# User defined parameters -- same as DiscoCH.ipynb
# Paths are machine-specific and come from config.yaml in the project root
# (copy config.example.yaml to config.yaml and set the paths for your machine).
from datetime import date
from src.disco_ch_operational.config import load_config

cfg = load_config()
force_directory = cfg["force_dir"]            # Local FORCE VI data (synced by DiscoCH.ipynb)
output_dir = cfg["output_dir"]                # Where to save output data
force_tiles_to_process = cfg["tiles_csv"]     # Tiles to process (Tile_ID column)
disco_model = cfg["model_path"]               # Pre-trained Discoloration Model
mask_path = cfg["mask_path"]                  # Forest mask (EPSG:2056, 1 = keep)
year = 2026

# Same RBF settings and VI naming as the near-real-time run
rbf_sigma = (10, 20, 30)
vi_keys = {
    "NDVI": "NDVI",
    "EVI": "EVI",
    "NDMI": "NDMI",
    "CIRE": "CIre",
    "CCI": "CCI",
}

In [ ]:
# Libraries
import os
import matplotlib.pyplot as plt

from src.disco_ch_operational.interpolate import run_full_data_season
from src.disco_ch_operational.mask import mask_and_scale_dir
from src.disco_ch_operational.compare import plot_year_comparison, BIOGEO_NAMES_EN, BIOGEO_COLORS

# Reset to the standard light theme for plots
plt.style.use('default')

# Explicitly force figures to white:
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# Folders: near-real-time (from DiscoCH.ipynb) and all-data (written here)
force_output = os.path.join(output_dir, 'DiscoCH_rbf_10_20_30')
nrt_output = os.path.join(force_output, str(year))
nrt_masked = os.path.join(nrt_output, 'masked')
full_rbf_output = os.path.join(force_output, 'level4_rbf_full_data', str(year))
full_output = os.path.join(force_output, f'{year}_full_data')
full_masked = os.path.join(full_output, 'masked')
stats_output = os.path.join(full_output, 'statistics')

## Interpolate with all data and apply the model

In [ ]:
# 1) RBF with every observation (as_of=None), 2) model normalized with the
# near-real-time min/max of each date. Re-running skips finished files/dates.
interpolated, summary = run_full_data_season(
    tss_dir=force_directory,
    rbf_dir=full_rbf_output,
    out_dir=full_output,             # tiles, mosaic/, logs/
    nrt_out_dir=nrt_output,          # its minmax_by_date/ gives each date's min/max
    year=year,
    model_path=disco_model,
    vi_keys=vi_keys,
    date_range=(date(year, 1, 1), date(year, 12, 31)),  # same 5-day grid as the near-real-time run
    rbf_sigma=rbf_sigma, rbf_cutoff=0.95,
    tiles_csv=force_tiles_to_process,
    model_months=(6, 7, 8, 9),
    max_workers=10,
)

## Masking

In [ ]:
# Same mask and scaling as the near-real-time mosaics
status = mask_and_scale_dir(input_dir=os.path.join(full_output, 'mosaic'),
                            mask_path=mask_path,
                            output_dir=full_masked,
                            pattern='Disco_Proba_*.tif',
                            threads=4,
                            )

status = mask_and_scale_dir(input_dir=os.path.join(nrt_output, 'mosaic'),
                            mask_path=mask_path,
                            output_dir=nrt_masked,
                            pattern='Disco_Proba_*.tif',
                            threads=4,
                            )

## Spatiotemporal Statistics

In [ ]:
# Same plot as DiscoCH.ipynb, with the all-data series added
mosaics_2018 = cfg.require("mosaics_2018")
threshold = 4000  # Stored value (0-10000) at/above which a pixel is discolored: 4000 = 40 % probability

# Greyscale, also used by the regional plot: NRT solid grey, retrospective solid black, 2018 dashed black
nrt_label, retro_label, label_2018 = f"{year} (Near-Real-Time)", f"{year} (Retrospective)", "2018 (Retrospective)"
dataset_colors = {nrt_label: "0.6", retro_label: "black", label_2018: "black"}
dataset_styles = {nrt_label: "-", retro_label: "-", label_2018: "--"}

comparison = plot_year_comparison(
    {
        nrt_label: dict(mosaic_dir=nrt_masked, year=year),
        retro_label: dict(mosaic_dir=full_masked, year=year),
        label_2018: dict(mosaic_dir=mosaics_2018, pattern="Proba_mosaic_*.tif", year=2018),
    },
    colors=dataset_colors, linestyles=dataset_styles,
    threshold=threshold,
    output_dir=stats_output,
    output_name=f"discoloration_comparison_{year}_nrt_vs_full_data",
    max_workers=8,
)

In [ ]:
# Difference per date: all data minus near real time
total = comparison[comparison.zone == "All"].pivot(index="date", columns="dataset", values="proportion")
diff = (total[f"{year} (Retrospective)"] - total[f"{year} (Near-Real-Time)"]).dropna()
print(diff.rename("retrospective - near-real-time (proportion)").round(4).to_string())

### Per biogeographic region: 2026 vs. 2018
One colour per region (`DERegionNa`, in English, polygons with the same name merged; "All" = Switzerland in black); retrospective of both years: 2026 solid, 2018 dashed. Both years are counted per region here, so this takes a while.

In [ ]:
import geopandas as gpd

# UTF-8 as its .CPG says (the default); encoding="cp1252" would garble the umlauts
biogeo_regions = gpd.read_file(cfg.require("biogeo_regions"))
mosaics_2018 = cfg.require("mosaics_2018")
threshold = 4000  # 4000 = 40 % probability

by_region = plot_year_comparison(
    {
        retro_label: dict(mosaic_dir=full_masked, year=year),
        label_2018: dict(mosaic_dir=mosaics_2018, pattern="Proba_mosaic_*.tif", year=2018),
    },
    zones=biogeo_regions, zone_field="DERegionNa",   # regions = All + every DERegionNa
    region_names=BIOGEO_NAMES_EN,                   # German -> English names
    region_colors=BIOGEO_COLORS,                    # colour per region (and legend order)
    colors=dataset_colors, linestyles=dataset_styles,  # same as the plot above
    layout="overlay",                               # or "panels": one panel per region
    threshold=threshold,
    output_dir=stats_output,
    output_name=f"discoloration_comparison_{year}_vs_2018_biogeo_regions",
    max_workers=8,
)

### Maps of every date: Hardwald
One column per month, one row per date, on the grey swisstopo map. The shapefile only sets the extent (nothing is clipped, no outline); colours dark green (0) to yellow (1).

In [ ]:
from src.disco_ch_operational.plot_date_grid import plot_date_grid

example_extent = cfg["example_extent"]   # data/example_extents/Herbetswil.shp

for dataset in ("nrt", "retrospective"):   # near real time (<year>/masked), all data (<year>_full_data/masked)
    plot_date_grid(
        dataset=dataset,
        extent=example_extent,                   # extent only: no clipping, no outline (outline=True draws it)
        year=year,
        months=(6, 7, 8, 9),
        alpha=0.6,                         # opacity of the probability (0-1)
        min_proba=0,                     # lower probabilities not drawn; None = all
        output=os.path.join(stats_output, f"date_grid_herbetswil_{dataset}.png"),
    )